# 第4章 pandas数据分析 · 课堂代码

> 本 notebook 与课件《Python金融数据分析 · 第4章 pandas数据分析》配套。
> 内容改编自《Python金融大数据分析（第2版）》第5章。

**使用说明**
- 点击单元格，按 `Shift + Enter` 运行；
- DataFrame 在 Notebook 中会以漂亮的表格形式显示。

## 1. 初识 pandas

NumPy 数组没有标签、只能装数字，而真实行情有日期、代码、缺失值。pandas 给数组加上行列标签，成为"数据表"。

- **Series**：一列带索引的数据（如某股票的收盘价序列）
- **DataFrame**：一张带行列标签的表（如 多股票×多日期 的价格表）

In [ ]:
import numpy as np
import pandas as pd        # 约定俗成，照抄
print(pd.__version__)

## 2. Series：带标签的一列数据

左边是**索引**（标签），右边是值。不指定索引时默认为 0,1,2,...

In [ ]:
s = pd.Series([1500.35, 128.60, 55.20],
              index=["600519", "000858", "601318"])
s

Series 支持 NumPy 式向量化运算与布尔筛选。

**关键概念：对齐** —— 两个 Series 运算时按**索引对齐**而不是按位置，顺序打乱也不会错。

In [ ]:
print(s["600519"])               # 1500.35  按标签取值
print(s[["600519", "601318"]])   # 取多个
print(s * 1.05)                  # 所有值涨5%（向量化）
print(s[s > 100])                # 筛选出大于100的

In [ ]:
# 体会"按索引对齐"：两个顺序不同的 Series 相乘
weights = pd.Series([0.5, 0.3, 0.2], index=["601318", "600519", "000858"])
print(s * weights)     # 每个价格只与"自己代码"的权重相乘
print((s * weights).sum())   # 组合的加权价格

## 3. DataFrame：金融数据的"主战场"

字典的**键**变成列名。约定俗成的方向：**行=日期，列=资产**。

In [ ]:
data = {"茅台": [1500.0, 1520.0, 1515.0],
        "五粮液": [128.0, 129.5, 127.0],
        "平安": [55.0, 54.5, 56.0]}
df = pd.DataFrame(data,
                  index=["2026-07-01", "2026-07-02", "2026-07-03"])
df

拿到任何新数据，先把这些"体检动作"做一遍。`describe()` 相当于免费的初步报告。

In [ ]:
print(df.shape)       # (3, 3)      行数、列数
print(df.columns)     # 列名
print(df.index)       # 行索引（这里是日期）

In [ ]:
df.head(2)            # 前2行（head()默认5行）

In [ ]:
df.describe()         # 每列的统计摘要：均值、标准差、分位数等

## 4. 数据选取与筛选

- `df[...]` 里放列名：取**列**；
- 想按行取，用位置切片或 `loc/iloc`；
- **最常见的困惑**：`df["茅台"]` 可以，`df["2026-07-01"]` 却报错，因为 `[]` 默认针对列。

In [ ]:
print(df["茅台"])            # 取一列 -> Series
df[["茅台", "平安"]]          # 取多列 -> DataFrame

In [ ]:
df[0:2]          # 前两行（位置切片，含头不含尾）

**loc 按标签选；iloc 按位置选。**先想清楚"我要按名字找还是按位置找"，再选工具。

In [ ]:
print(df.loc["2026-07-02"])              # 该行全部列
print(df.loc["2026-07-02", "茅台"])      # 指定行、指定列 -> 1520.0
print(df.loc[:, "茅台"])                 # 所有行、茅台列

In [ ]:
print(df.iloc[0])             # 第0行（按位置）
print(df.iloc[0:2, 1])        # 前2行、第1列
print(df.iloc[-1])            # 最后一行

条件筛选与 NumPy 布尔索引完全一致：多条件组合用 `&`、`|`，每个条件加括号。

**课堂练习**：用一行代码选出"五粮液价格低于128"的所有交易日。

In [ ]:
df[df["茅台"] > 1510]       # 茅台价格高于1510的日子

In [ ]:
df[(df["茅台"] > 1500) & (df["平安"] > 55)]

In [ ]:
# 课堂练习：在这里写下你的代码


<details><summary>参考答案（先自己动手！）</summary>

```python
df[df["五粮液"] < 128]
```
</details>

## 5. 排序与分组

排序返回**新表**，原表不变。时间序列数据到手先 `sort_index()`，保证日期升序。

In [ ]:
df.sort_values("茅台", ascending=False)  # 按茅台价格降序

In [ ]:
df.sort_index()                          # 按行索引(日期)排序

**groupby 三步走**：分组（groupby）→ 选列 → 聚合（mean/sum/max...）。回答"每个行业的平均市盈率"这类问题，一行搞定。

In [ ]:
stocks = pd.DataFrame({
    "代码":  ["600519", "000858", "601318", "600036"],
    "行业":  ["白酒", "白酒", "保险", "银行"],
    "市盈率": [28.5, 18.2, 8.1, 6.3]})
stocks

In [ ]:
stocks.groupby("行业")["市盈率"].mean()

In [ ]:
# 也可以一次算多个统计量
stocks.groupby("行业")["市盈率"].agg(["mean", "max", "count"])

## 6. 金融时间序列

### 6.1 把字符串变成日期

`pd.to_datetime()` 把字符串转成真正的日期类型；之后可以按年、月智能取值：`df["2026-07"]` 直接取出2026年7月的数据。`date_range` 生成规则日期序列，`freq="B"` 表示工作日。

In [ ]:
df.index = pd.to_datetime(df.index)
df.index       # DatetimeIndex

In [ ]:
pd.date_range("2026-01-01", periods=5, freq="B")
# 从1月1日起的5个工作日（B = Business day）

### 6.2 收益率计算：pct_change 与 shift

- `pct_change()`：把第3章的手写公式封装成方法，整张表所有列**一次全算**；
- `shift(1)`：错位是金融计算的核心技巧——"昨日收益""信号与次日收益配对"都靠它；
- 第一行没有昨日数据，结果自动为 NaN。

In [ ]:
df.pct_change()        # 每列的日收益率：(今日-昨日)/昨日

In [ ]:
print(df.shift(1))     # 整表向下错位一行 -> 昨日价格
print(df.diff())       # 价格变动额（今日-昨日）

### 6.3 重采样：日频转月频

`resample(频率)` 像"汇总报表"，后面必须接聚合方式（last/mean/sum）。常用频率：`"D"` 日、`"W"` 周、`"ME"` 月末、`"YE"` 年末。

In [ ]:
df.resample("ME").last()    # 每月最后一个交易日的价格

## 7. 缺失值处理

NaN 表示缺失：停牌、数据未披露、`pct_change` 的第一行。`ffill()`（前向填充）是价格序列中最自然的补法。处理前先想：缺失代表什么经济含义？

In [ ]:
s2 = pd.Series([1500.0, np.nan, 1515.0])

print(s2.isna())             # [False, True, False]  找出缺失
print(s2.dropna())           # 删掉缺失的行
print(s2.fillna(s2.mean()))  # 用均值填充
print(s2.ffill())            # 用前一个有效值填充（金融常用！）

## 8. 读写数据

真实数据多以 CSV/Excel 形式存在。读入时顺手完成"第一列当日期索引"：

```python
df = pd.read_csv("eod_data.csv", index_col=0, parse_dates=True)
```

下面先把手头的表导出为 CSV，再读回来，体验完整的往返流程。

In [ ]:
df.to_csv("demo_prices.csv")                       # 导出
df2 = pd.read_csv("demo_prices.csv",               # 读回
                  index_col=0,                     # 第0列作为索引
                  parse_dates=True)                # 解析为日期
print(type(df2.index))
df2

## 9. 综合案例：三股风险分析

构造 250 个交易日的三只股票价格，完成"收益率—风险—相关性"分析。**一个方法作用于所有列**，全程没有循环。

In [ ]:
rng = np.random.default_rng(42)
dates = pd.date_range("2026-01-05", periods=250, freq="B")

# 模拟三只股票的日收益率，再还原为价格
rets = pd.DataFrame({
    "茅台": 0.0004 + 0.015 * rng.standard_normal(250),
    "平安": 0.0002 + 0.020 * rng.standard_normal(250),
    "招行": 0.0003 + 0.018 * rng.standard_normal(250)},
    index=dates)
prices = 100 * np.exp(rets.cumsum())
prices.tail(3)

In [ ]:
rets = prices.pct_change().dropna()   # 日收益率

rets.describe().round(4)              # 统计摘要

In [ ]:
print("年化平均收益：")
print((rets.mean() * 252).round(4))

print("年化波动率：")
print((rets.std() * np.sqrt(252)).round(4))

In [ ]:
rets.corr().round(2)                  # 相关系数矩阵

**课堂讨论**：相关系数接近 1 的两只股票，能互相分散风险吗？

## 10. 课后任务

1. 运行本 notebook，替换 `freq`、随机种子等参数观察输出；
2. 对案例数据：找出每只股票的最大单日跌幅及其日期（提示：`idxmin()` 返回最小值所在的索引）；
3. 选做：用 `resample("ME").last()` 得到月末价格，再算月度收益率。